In [1]:
import pandas as pd

df = pd.read_stata("chrisu.dta")  


print(df.shape)       # rows, columns
print(df.head())      # first 5 rows
print(df.columns)     # all variable names
df.info()             # variable types and missing values

(3530, 124)
   V1                         V2      V3                  V4  \
0  21  Asian or Pacific Islander  Female     Senior, college   
1  20                      White  Female     Junior, college   
2  19                      White  Female  Sophomore, college   
3  21                      White  Female     Junior, college   
4  20                      White  Female   Freshman, college   

                     V5               V6          V7            V8  \
0       Graduate degree  Graduate degree        None  Five or more   
1       Graduate degree  Graduate degree        None  Five or more   
2       Graduate degree  Graduate degree  One tattoo          Four   
3  Some graduate school   College degree        None  Five or more   
4       Graduate degree  Graduate degree        None  Five or more   

                  V9                V10  ...  V115 V116 V117 V118 V119  V120  \
0                One  None have tattoos  ...   NaN  NaN  NaN  NaN  NaN   NaN   
1  None have tattoos  

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_stata("chrisu.DTA", convert_categoricals=False)

# 1 = any illegal drug use, 0 = none
df["drug_use"] = np.where(df["V79"].isna(), np.nan, (df["V79"] > 0).astype(int))

print(df["drug_use"].value_counts(dropna=False))   # expect 3221 zeros, 230 ones, 79 missing

drug_use
0.0    3221
1.0     230
NaN      79
Name: count, dtype: int64


/var/folders/66/ntlk8n594_9346y_qj6fmv0c0000gn/T/ipykernel_68551/4211556772.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["drug_use"] = np.where(df["V79"].isna(), np.nan, (df["V79"] > 0).astype(int))


In [3]:
vars_needed = ["drug_use", "V25", "V26", "V27", "V28", "V29", "V34"]
analysis = df[vars_needed].dropna().copy()
print("Original N:", len(df), "| Analysis N:", len(analysis))

model = smf.logit("drug_use ~ V25 + V26 + V27 + V28 + V29 + V34", data=analysis).fit()
print(model.summary())

# odds ratios with confidence intervals
import numpy as np
odds = pd.DataFrame({
    "OR": np.exp(model.params),
    "CI_low": np.exp(model.conf_int()[0]),
    "CI_high": np.exp(model.conf_int()[1]),
    "p": model.pvalues,
})
print(odds.round(3))

Original N: 3530 | Analysis N: 3059
Optimization terminated successfully.
         Current function value: 0.222867
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               drug_use   No. Observations:                 3059
Model:                          Logit   Df Residuals:                     3052
Method:                           MLE   Df Model:                            6
Date:                Thu, 01 Oct 2026   Pseudo R-squ.:                 0.04242
Time:                        10:37:54   Log-Likelihood:                -681.75
converged:                       True   LL-Null:                       -711.95
Covariance Type:            nonrobust   LLR p-value:                 3.742e-11
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.9699      0.127    -15.471      0.000      -2.219      

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_stata("chrisu.DTA", convert_categoricals=False)

# 1. Rename
df = df.rename(columns={
    "V25": "pray_freq",          # how often do you pray
    "V26": "table_grace",        # table prayers / grace
    "V27": "pray_self",          # pray about own needs
    "V28": "pray_close",         # pray about people close to you
    "V29": "pray_others",        # pray about others in general
    "V79": "drug_freq",          # raw 0-3 frequency of illegal drug use
})

# 2. Recode the outcome into 0/1 (keeps missing as missing)
df["drug_use"] = np.where(df["drug_freq"].isna(), np.nan, (df["drug_freq"] > 0).astype(int))

# 3. Analysis sample
vars_needed = ["drug_use", "pray_freq", "table_grace", "pray_self",
               "pray_close", "pray_others"]
analysis = df[vars_needed].dropna().copy()
print("Original N:", len(df), "| Analysis N:", len(analysis))

# 4. Model, now with readable names
model1 = smf.logit(
    "drug_use ~ pray_freq + table_grace + pray_self + pray_close + pray_others",
    data=analysis
).fit()
print(model1.summary())

Original N: 3530 | Analysis N: 3421
Optimization terminated successfully.
         Current function value: 0.235370
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               drug_use   No. Observations:                 3421
Model:                          Logit   Df Residuals:                     3415
Method:                           MLE   Df Model:                            5
Date:                Thu, 01 Oct 2026   Pseudo R-squ.:                 0.03272
Time:                        10:37:54   Log-Likelihood:                -805.20
converged:                       True   LL-Null:                       -832.44
Covariance Type:            nonrobust   LLR p-value:                 1.669e-10
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -2.0802      0.107    -19.523      0.000      -2.289   

/var/folders/66/ntlk8n594_9346y_qj6fmv0c0000gn/T/ipykernel_68551/4201849979.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["drug_use"] = np.where(df["drug_freq"].isna(), np.nan, (df["drug_freq"] > 0).astype(int))


In [5]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_stata("chrisu.DTA", convert_categoricals=False)

# rename numbered vairables to what they actually measure for easy reference
df = df.rename(columns={
    "V25": "pray_freq",          # how often do you pray
    "V26": "table_grace",        # table prayers / grace
    "V27": "pray_self",          # pray about own needs
    "V28": "pray_close",         # pray about people close to you
    "V29": "pray_others",        # pray about others in general
    "V34": "attend_services",    # attendance at religious services
    "V79": "drug_freq",          # raw 0-3 frequency of illegal drug use
})

# recode the dependent vairable into 0/1 --> binary (keeps missing as missing)
df["drug_use"] = np.where(df["drug_freq"].isna(), np.nan, (df["drug_freq"] > 0).astype(int))

vars_needed = ["drug_use", "pray_freq", "table_grace", "pray_self",
               "pray_close", "pray_others", "attend_services"]
analysis = df[vars_needed].dropna().copy()
print("Original N:", len(df), "| Analysis N:", len(analysis))

model2 = smf.logit(
    "drug_use ~ pray_freq + table_grace + pray_self + pray_close + pray_others + attend_services",
    data=analysis
).fit()
print(model2.summary())

/var/folders/66/ntlk8n594_9346y_qj6fmv0c0000gn/T/ipykernel_68551/3332734030.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["drug_use"] = np.where(df["drug_freq"].isna(), np.nan, (df["drug_freq"] > 0).astype(int))


Original N: 3530 | Analysis N: 3059
Optimization terminated successfully.
         Current function value: 0.222867
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               drug_use   No. Observations:                 3059
Model:                          Logit   Df Residuals:                     3052
Method:                           MLE   Df Model:                            6
Date:                Thu, 01 Oct 2026   Pseudo R-squ.:                 0.04242
Time:                        10:37:54   Log-Likelihood:                -681.75
converged:                       True   LL-Null:                       -711.95
Covariance Type:            nonrobust   LLR p-value:                 3.742e-11
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -1.9699      0.127    -15.471      0.000   

In [6]:
from statsmodels.iolib.summary2 import summary_col

In [7]:
table = summary_col(
    [model1, model2],
    model_names=["Model 1", "Model 2"],
    stars=True,
    float_format="%.3f",
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "Pseudo R²": lambda x: f"{x.prsquared:.3f}",
        "Log-Likelihood": lambda x: f"{x.llf:.1f}",
        "AIC": lambda x: f"{x.aic:.1f}",
    },
)
print(table)


                 Model 1   Model 2 
-----------------------------------
Intercept       -2.080*** -1.970***
                (0.107)   (0.127)  
pray_freq       -0.292*** -0.207** 
                (0.080)   (0.087)  
table_grace     -0.113    -0.071   
                (0.085)   (0.093)  
pray_self       0.172*    0.259**  
                (0.102)   (0.107)  
pray_close      -0.195*   -0.258** 
                (0.109)   (0.115)  
pray_others     0.105     0.132    
                (0.094)   (0.098)  
attend_services           -0.190***
                          (0.064)  
AIC             1622.4    1377.5   
Log-Likelihood  -805.2    -681.8   
N               3421      3059     
Pseudo R²       0.033     0.042    
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01


this is to ensure the number of survey answers used to analyse are the same for each model

In [8]:
from stargazer.stargazer import Stargazer

sg = Stargazer([model1, model2])

sg.dependent_variable_name('Drug Usage')
sg.custom_columns(['Model 1', 'Model 2'], [1, 1])

# 1. Order using ORIGINAL names
sg.covariate_order([
    'pray_freq', 'table_grace', 'pray_self', 'pray_close',
    'pray_others', 'attend_services', 'Intercept'
])

# 2. Rename to readable labels
sg.rename_covariates({
    'pray_freq':       'Praying Frequency',
    'table_grace':     'Saying Grace',
    'pray_self':       'Praying for Self',
    'pray_close':      'Praying for Close People',
    'pray_others':     'Praying for Others',
    'attend_services': 'Attending Services',
    'Intercept':       'Intercept',
})

sg

Model 1: prayer frequency is significantly and negatively associated with drug use (beta coefficient = - 0.292, p<0.01). one unit increase in prayer frequency is expected to cause an decrease of log odds of drug usage by 0.287 (p<0.01), ceteris paribus. prayer frequency has the largest coefficient among the predictors in model 1, suggesting that how often students prayer is the religious practice measure most strongly linked to lower drug usage. given a small difference in coeffeicinet value for praying for close people the p-value of this variable = 0.115 > 0.1, hence, null hypothesis cannot be rejected, results are statistically insignificant. 

Model 2: adding the independent variable of attending services in model 2 lowers the associated effect of prayer on drug usage frequency by log odds of 0.080, (p<0.01), ceteris paribus. however, the variable of praying frequency still remains most strongly linked to lower drug usage amon religious practices measured in this survey.    

In [9]:
print(f'AIC logit_model1: {model1.aic:.2f}')
print(f'AIC logit_model2: {model2.aic:.2f}')

AIC logit_model1: 1622.40
AIC logit_model2: 1377.50


Model 2 with an additional variable is a better-fit model as the AIC value decreases greatly by more than 200, and a smaller value signals a better fit gicen that AIC adds a small penalty for each variable added.

from here on: finding spearman's correlation between vairables that involve praying to explore possibility of overlap

In [10]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# 1. Load the data
df = pd.read_stata("chrisu.DTA", convert_categoricals=False)

# 2. Rename variables
df = df.rename(columns={
    "V25": "pray_freq",
    "V26": "table_grace",
    "V27": "pray_self",
    "V28": "pray_close",
    "V29": "pray_others",
    "V34": "attend_services",
    "V79": "drug_freq",
})

# 3. Binary outcome: 1 = any illegal drug use, 0 = none, missing stays missing
df["drug_use"] = np.where(df["drug_freq"].isna(), np.nan, (df["drug_freq"] > 0).astype(int))

# 4. Analysis sample (complete cases on all variables)
cols = ["drug_use", "pray_freq", "table_grace", "pray_self",
        "pray_close", "pray_others", "attend_services"]
same = df[cols].dropna().copy()
print("N =", len(same))

# 5. Models
model1 = smf.logit("drug_use ~ pray_freq + table_grace + pray_self + pray_close + pray_others",
                   data=same).fit()
model2 = smf.logit("drug_use ~ pray_freq + table_grace + pray_self + pray_close + pray_others + attend_services",
                   data=same).fit()

# 6. Correlations (Spearman suits ordered categories)
same.corr(method="spearman").round(2)

N = 3059
Optimization terminated successfully.
         Current function value: 0.224338
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.222867
         Iterations 7


/var/folders/66/ntlk8n594_9346y_qj6fmv0c0000gn/T/ipykernel_68551/112925737.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["drug_use"] = np.where(df["drug_freq"].isna(), np.nan, (df["drug_freq"] > 0).astype(int))


,drug_use,pray_freq,table_grace,pray_self,pray_close,pray_others,attend_services
drug_use,1.00,-0.11,-0.09,-0.07,-0.10,-0.07,-0.12
pray_freq,-0.11,1.00,0.61,0.76,0.71,0.67,0.68
table_grace,-0.09,0.61,1.00,0.51,0.49,0.48,0.61
pray_self,-0.07,0.76,0.51,1.00,0.79,0.67,0.58
pray_close,-0.10,0.71,0.49,0.79,1.00,0.76,0.56
pray_others,-0.07,0.67,0.48,0.67,0.76,1.00,0.52
attend_services,-0.12,0.68,0.61,0.58,0.56,0.52,1.00


shows high correlation i.e. > 0.70 (common rule of thumb) between some variables (e.g. pray_self & pray_freq), and would be better to combine such variables 

In [11]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

items = ["pray_freq", "table_grace", "pray_self", "pray_close", "pray_others", "attend_services"]
threshold = 0.70      # what counts as "highly correlated"

# 1. Find items with at least one high correlation (|r| >= threshold) with another item
corr = same[items].corr(method="spearman")
high = [c for c in items
        if ((corr[c].drop(c)).abs() >= threshold).any()]
low = [c for c in items if c not in high]
print("Combined into index:", high)
print("Kept separate:      ", low)

# 2. Standardise (z-scores) so items on 0-4 and 0-5 scales count equally, then average
z = (same[high] - same[high].mean()) / same[high].std()
same["religiosity_index"] = z.mean(axis=1)

# 3. Reliability check (Cronbach's alpha; 0.7+ is acceptable)
k = len(high)
alpha = k / (k - 1) * (1 - z.var(ddof=1).sum() / z.sum(axis=1).var(ddof=1))
print("Cronbach's alpha:", round(alpha, 3))

# 4. Models using the index plus the items left separate
formula = "drug_use ~ religiosity_index" + "".join(f" + {c}" for c in low)
model_idx = smf.logit(formula, data=same).fit()
print(model_idx.summary())

# 5. Odds ratios
print(np.exp(model_idx.params).round(3))

Combined into index: ['pray_freq', 'pray_self', 'pray_close', 'pray_others']
Kept separate:       ['table_grace', 'attend_services']
Cronbach's alpha: 0.907
Optimization terminated successfully.
         Current function value: 0.224764
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               drug_use   No. Observations:                 3059
Model:                          Logit   Df Residuals:                     3055
Method:                           MLE   Df Model:                            3
Date:                Thu, 01 Oct 2026   Pseudo R-squ.:                 0.03427
Time:                        10:37:54   Log-Likelihood:                -687.55
converged:                       True   LL-Null:                       -711.95
Covariance Type:            nonrobust   LLR p-value:                 1.443e-10
                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------

In [12]:
from stargazer.stargazer import Stargazer

sg = Stargazer([model2, model_idx])

sg.dependent_variable_name('Drug Usage')
sg.custom_columns(['Item-by-item (Model 2)', 'With Prayer Index'], [1, 1])

# 1. Order using ORIGINAL names (index first, then the individual items)
sg.covariate_order([
    'religiosity_index',
    'pray_freq', 'pray_self', 'pray_close', 'pray_others',
    'table_grace', 'attend_services',
    'Intercept'
])

# 2. Readable labels
sg.rename_covariates({
    'religiosity_index': 'Prayer Index (SD units)',
    'pray_freq':         'Praying Frequency',
    'pray_self':         'Praying for Self',
    'pray_close':        'Praying for Close People',
    'pray_others':       'Praying for Others',
    'table_grace':       'Saying Grace',
    'attend_services':   'Attending Services',
    'Intercept':         'Intercept',
})

sg